# Deployment les 1 - sklearn pipelines

Notebook bij les 1 van de leerlijn deployment van S3 - AI. In dit notebook wordt eerst een voorbeeld gegeven van de werking van het object sklearn pipelines in een training pipeline en inference pipeline. Het tweede deel bestaat uit een aantal opdrachten.

© Auteur: Joost Vanstreels

**Voorbereiding voor les 1:** Bekijk het voorbeeld (deel 1 van dit notebook), zorg dat je alle cellen kunt runnen én dat je snapt wat er gebeurt.

## 0. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import FunctionTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

import joblib


# Deel 1. Voorbeeld Training & Inference pipeline

We maken onderscheid tussen een training pipeline (waarin het doel is om een model te trainen) en een inference pipeline (waarin het model toegepast wordt op nieuwe data). Hieronder wordt de training pipeline in 8 stappen opgesteld.

## 1. Dataset maken

In dit notebook maken we gebruik van een kleine, fictieve dataset. We voegen bewust enkele problemen toe die in echte datasets kunnen voorkomen:

- ontbrekende leeftijd;
- ontbrekende studie-uren;
- inconsistente categorie (`ict` versus `ICT`);
- een aantal outliers.

In [ ]:
data = pd.DataFrame({
    "leeftijd": [19, 21, 22, 20, 25, 23, 24, 19, 21, 22, 27, 31, 110],
    "studie_uren": [10, 15, 12, 8, 20, 18, 14, 9, 16, 13, 22, 80, 12],
    "opleiding": ["ICT", "ICT", "Business", "ICT", "Business", "ICT",
                  "ICT", "Business", "ict", np.nan, "Business", "ICT", "ICT"],
    "geslaagd": [0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0]
})

data.loc[3, "leeftijd"] = np.nan
data.loc[7, "studie_uren"] = np.nan

data


## 2. Eerste inspectie

In [ ]:
print(data.info())
print("\nOntbrekende waarden:")
print(data.isna().sum())

print("\nCategorieën:")
print(data["opleiding"].value_counts(dropna=False))

print("\nNumerieke beschrijving:")
display(data.describe())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].boxplot(data["leeftijd"].dropna())
axes[0].set_title("Leeftijd")
axes[0].set_ylabel("Leeftijd")

axes[1].boxplot(data["studie_uren"].dropna())
axes[1].set_title("Studie-uren")
axes[1].set_ylabel("Uren per week")

plt.show()


## 3. Outliers expliciet onderzoeken

De twee opvallende waarden zijn:

- **110 jaar**;
- **80 studie-uren**.

We bekijken ze eerst en verwijderen ze niet zomaar.

In [ ]:
outlier_age = data["leeftijd"] > 100
outlier_study = data["studie_uren"] > 50

print("Personen ouder dan 100:")
display(data[outlier_age])

print("Personen met meer dan 50 studie-uren:")
display(data[outlier_study])


Conclusie:
- **110 jaar**: deze verwijderen we omdat het waarschijnlijk een invoerfout is.
- **80 studie-uren**: extreme maar mogelijke waarde en **behouden**.

In [ ]:
data_clean = data.loc[~outlier_age].copy()

print("Aantal rijen vóór:", len(data))
print("Aantal rijen na :", len(data_clean))
display(data_clean)


## 4. X en y

We gaan eerst de data splitten voordat we *iets* gaan doen met de NA's.

In [ ]:
X = data_clean[["leeftijd", "studie_uren", "opleiding"]]
y = data_clean["geslaagd"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print("Training:", X_train.shape)
print("Test:", X_test.shape)


## 5. Preprocessing

We splitsen de data op in numerieke kolommen en categorische kolommen en maken  twee preprocessing-routes. Dat is om de pipeline overzichtelijk te houden maar ook omdat er verschillende acties bestaan voor beide soorten kolommen.

### Numeriek

- `SimpleImputer` → ontbrekende waarden (NA's) vullen door de mediaan.
- `StandardScaler` → kolommen schalen.

### Categorisch

De historische data bevat zowel `"ICT"` als `"ict"`. We weten dat dit dezelfde opleiding betekent.

Daarom maken we binnen de **categorical pipeline** eerst:

```text
"ict" → "ICT"
```

Daarna:

- `SimpleImputer` → ontbrekende categorieën vullen;
- `OneHotEncoder` → categorieën omzetten naar numerieke kolommen.

In [ ]:
# In dit codeblok worden functies gedefinieerd die in de pipeline uitgevoerd moeten worden

# Splits de kolommen in numerieke features en categoriale features omdat deze op een andere manier 'gepreprocessed' worden
numeric_features = ["leeftijd", "studie_uren"]
categorical_features = ["opleiding"]

# clean_opleiding is een custom functie voor de pipeline
def clean_opleiding(X):
    X = X.copy()
    X["opleiding"] = X["opleiding"].replace({
        "ict": "ICT"
    })
    return X

# Daarnaast maken we gebruik van een aantal standaard functies
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("clean_opleiding", FunctionTransformer(
        clean_opleiding,
        validate=False,
        feature_names_out="one-to-one"
    )),
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

### Let op de hiërarchie van de objecten!

Zie hieronder welke objecten in de pipeline zitten:

```text
ColumnTransformer
│
├── numeric
│   └── Pipeline
│       └── Imputer
│       └── StandardScaler
│
└── categorical
    └── Pipeline
        ├── clean_opleiding
        ├── Imputer
        └── OneHotEncoder
```

De `ColumnTransformer` bepaalt **welke kolommen** welke route volgen.

De `Pipeline` binnen de categorieën bepaalt **welke stappen achter elkaar** worden uitgevoerd.

Voor `opleiding` moeten dus eerst meerdere stappen gebeuren:

```text
"ict"
  ↓
"ICT"
  ↓
missing values oplossen
  ↓
one-hot encoding
```

Dit is precies het soort hiërarchie waarvoor `ColumnTransformer` en `Pipeline` samen bedoeld zijn.

Het resultaat van de preprocessor kun je ook bekijken. Je kunt de data *handmatig* door de preprocessor leiden om het resultaat te bekijken. Dat is een numpy array omdat modellen *onder water* geen dataframes maar arrays gebruiken.

In [ ]:
X_train_processed = preprocessor.fit_transform(X_train)
X_train_processed

De namen van de features kun je ook bekijken. Dat is handig om te controleren of alles goed gegaan is en er niet per ongeluk een kolom ```categorical__opleiding_ict``` gemaakt is.

In [ ]:
feature_names = preprocessor.get_feature_names_out()
feature_names

## 6. Model + preprocessing combineren

In de uiteindelijke pipeline gaan we ook het model opnemen. In dit geval een DecisionTreeClassifier.

**Merk op**: hierboven hebben we de een nieuwe versie van X_train gemaakt die je kunt gebruiken voor het trainen van je modellen. Dat hoeft niet persé: je kunt het trainen van je model ook in een pipeline doen. Dan regelt de pipeline zelf dat de originele data getransformeerd wordt voordat modellen getraind worden.

In [ ]:
model_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", DecisionTreeClassifier(max_depth=3, random_state=42))
])

model_pipeline.fit(X_train, y_train)


## 7. Evalueren
Normaal gesproken moest je bij het evalueren met de testset, nóg een keer alle preparatie stappen uitvoeren op de testset. Dat moet nog steeds, maar het is wel een stuk makkelijker want je kunt *gewoon* de pipeline gebruiken.

In [ ]:
# Hieronder zie je ten eerste het resultaat van de preprocessing op de dataset `X`
# Dit is het numpy array dat als input gegeven is aan het model
# Je ziet dat de eerste twee kolommen genormaliseerd zijn en dat de laatste twee kolommen dummies zijn
preprocessor = model_pipeline.named_steps["preprocessing"]
preprocessor.transform(X)

# Wat is er gebeurd met de velden waar np.nan stond?

In [ ]:
# Hieronder zie je het resultaat van de hele pipeline (dus preprocessing en model) op de dataset `X_test`
# Let op: je hoeft nu dus niet de fit te doen
y_pred = model_pipeline.predict(X_test)

ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
plt.show()


## 8. Volledige Pipeline exporteren

We slaan de **volledige Pipeline** op in een joblib object. Dit object kun je in een ander notebook, een andere applicatie of Streamlit dashboard importeren.

In [ ]:
MODEL_PATH = "model_pipeline.joblib"

joblib.dump(model_pipeline, MODEL_PATH)

print(f"Pipeline opgeslagen als: {MODEL_PATH}")


# Inference pipeline

De code hieronder simuleert wat er in de deployment gebeurt: een andere applicatie of Streamlit gebruikt alleen het opgeslagen bestand. De *inference pipeline* wordt toegepast op nieuwe data en geeft een voorspelling.

Je ziet dat er geen losse stappen meer nodig zijn, het is feitelijk gewoon één functionele stap: het model aanroepen met nieuwe data.

In [ ]:
# Start met het importeren van de pipeline
MODEL_PATH = "model_pipeline.joblib"
loaded_pipeline = joblib.load(MODEL_PATH)

In [ ]:
# Nieuwe data
# Expres "ict" met kleine letters:
# de pipeline maakt hier automatisch "ICT" van en plaats deze in de correcte kolom.
nieuwe_studenten = pd.DataFrame({
    "leeftijd": [20, np.nan, 30],
    "studie_uren": [14, 80, 40],
    "opleiding": ["ict", "Business", "Design"]
})

# Kijk wat er gebeurt als je np.nan op andere plekken zet of niet bestaande studies gebruikt


In [ ]:
# En de daadwerkelijke inference van het model
voorspellingen = loaded_pipeline.predict(nieuwe_studenten)

resultaat = nieuwe_studenten.copy()
resultaat["voorspelling_geslaagd"] = voorspellingen
resultaat

In [ ]:
# Hieronder kun je zien wat het resultaat is van de preprocessor
preprocessor = loaded_pipeline.named_steps["preprocessing"]
preprocessor.transform(nieuwe_studenten)

# Wat valt op in de laatste twee dummy kolommen bij de laatste student?

# Deel 2. Opdrachten

## Opdracht 1.
In deze opdracht gaan we de zojuist gemaakte pipeline proberen te slopen. De pipeline is nu zodanig opgezet dat ie *hufterproof* is, maar dat gaan we nu ongedaan maken

### Opdracht 1.1.
Plaats in de ```categorical_pipeline``` de ```encoder``` als eerste actie, wat gaat er mis? Plaats 'm nu als tweede actie en check wat het effect is op ```nieuwe_studenten_processed```. Hoe komt dat?


In [ ]:
# DEZE CODE MOET JE AANPASSEN VOOR ELKE OPDRACHT

# Splits de kolommen in numerieke features en categoriale features omdat deze op een andere manier 'gepreprocessed' worden
numeric_features = ["leeftijd", "studie_uren"]
categorical_features = ["opleiding"]

# clean_opleiding is een custom functie voor de pipeline
def clean_opleiding(X):
    X = X.copy()
    X["opleiding"] = X["opleiding"].replace({
        "ict": "ICT"
    })
    return X

# Daarnaast maken we gebruik van een aantal standaard functies
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("clean_opleiding", FunctionTransformer(
        clean_opleiding,
        validate=False,
        feature_names_out="one-to-one"
    )),
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

# De pipeline wordt compleet gemaakt door de classifier
model_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", DecisionTreeClassifier(max_depth=3, random_state=42))
])

# De pipeline wordt uitgevoerd op de train data
model_pipeline.fit(X_train, y_train)

In [ ]:
# DEZE CODE GEBRUIK JE OM HET RESULTAAT TE TESTEN
preprocessor = model_pipeline.named_steps["preprocessing"]
nieuwe_studenten_processed = preprocessor.transform(nieuwe_studenten)
nieuwe_studenten_processed

In [ ]:
# DEZE CODE GEBRUIK JE OM HET RESULTAAT TE TESTEN
feature_names = preprocessor.get_feature_names_out()
feature_names

### Opdracht 1.2.
Verander bij de ```OneHotEncorder``` de parameter ```ignore``` naar ```error``` en naar ```infrequent_if_exist``` en check wat het effect is op ```nieuwe_studenten_processed```. Hoe komt dat?

In [ ]:
# DEZE CODE MOET JE AANPASSEN VOOR ELKE OPDRACHT

# Splits de kolommen in numerieke features en categoriale features omdat deze op een andere manier 'gepreprocessed' worden
numeric_features = ["leeftijd", "studie_uren"]
categorical_features = ["opleiding"]

# clean_opleiding is een custom functie voor de pipeline
def clean_opleiding(X):
    X = X.copy()
    X["opleiding"] = X["opleiding"].replace({
        "ict": "ICT"
    })
    return X

# Daarnaast maken we gebruik van een aantal standaard functies
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("clean_opleiding", FunctionTransformer(
        clean_opleiding,
        validate=False,
        feature_names_out="one-to-one"
    )),
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

# De pipeline wordt compleet gemaakt door de classifier
model_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", DecisionTreeClassifier(max_depth=3, random_state=42))
])

# De pipeline wordt uitgevoerd op de train data
model_pipeline.fit(X_train, y_train)

In [ ]:
# DEZE CODE GEBRUIK JE OM HET RESULTAAT TE TESTEN
preprocessor = model_pipeline.named_steps["preprocessing"]
nieuwe_studenten_processed = preprocessor.transform(nieuwe_studenten)
nieuwe_studenten_processed

In [ ]:
# DEZE CODE GEBRUIK JE OM HET RESULTAAT TE TESTEN
feature_names = preprocessor.get_feature_names_out()
feature_names

### Opdracht 1.3.
Verwijder bij de ```numerical_pipeline``` de ```imputer``` en check wat het effect is op ```nieuwe_studenten_processed```. Hoe komt dat?

In [ ]:
# DEZE CODE MOET JE AANPASSEN VOOR ELKE OPDRACHT

# Splits de kolommen in numerieke features en categoriale features omdat deze op een andere manier 'gepreprocessed' worden
numeric_features = ["leeftijd", "studie_uren"]
categorical_features = ["opleiding"]

# clean_opleiding is een custom functie voor de pipeline
def clean_opleiding(X):
    X = X.copy()
    X["opleiding"] = X["opleiding"].replace({
        "ict": "ICT"
    })
    return X

# Daarnaast maken we gebruik van een aantal standaard functies
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("clean_opleiding", FunctionTransformer(
        clean_opleiding,
        validate=False,
        feature_names_out="one-to-one"
    )),
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

# De pipeline wordt compleet gemaakt door de classifier
model_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", DecisionTreeClassifier(max_depth=3, random_state=42))
])

# De pipeline wordt uitgevoerd op de train data
model_pipeline.fit(X_train, y_train)

In [ ]:
# DEZE CODE GEBRUIK JE OM HET RESULTAAT TE TESTEN
preprocessor = model_pipeline.named_steps["preprocessing"]
nieuwe_studenten_processed = preprocessor.transform(nieuwe_studenten)
nieuwe_studenten_processed

In [ ]:
# DEZE CODE GEBRUIK JE OM HET RESULTAAT TE TESTEN
feature_names = preprocessor.get_feature_names_out()
feature_names

### Opdracht 1.4.
Pas de ```numerical_pipeline``` alleen toe op de kolom ```leeftijd``` en check wat het effect is op ```nieuwe_studenten_processed```.

Hoe kun je ervoor zorgen dat het aantal studie uren ook meegegeven wordt aan het model?

In [ ]:
# DEZE CODE MOET JE AANPASSEN VOOR ELKE OPDRACHT

# Splits de kolommen in numerieke features en categoriale features omdat deze op een andere manier 'gepreprocessed' worden
numeric_features = ["leeftijd", "studie_uren"]
categorical_features = ["opleiding"]

# clean_opleiding is een custom functie voor de pipeline
def clean_opleiding(X):
    X = X.copy()
    X["opleiding"] = X["opleiding"].replace({
        "ict": "ICT"
    })
    return X

# Daarnaast maken we gebruik van een aantal standaard functies
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("clean_opleiding", FunctionTransformer(
        clean_opleiding,
        validate=False,
        feature_names_out="one-to-one"
    )),
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

# De pipeline wordt compleet gemaakt door de classifier
model_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", DecisionTreeClassifier(max_depth=3, random_state=42))
])

# De pipeline wordt uitgevoerd op de train data
model_pipeline.fit(X_train, y_train)

In [ ]:
# DEZE CODE GEBRUIK JE OM HET RESULTAAT TE TESTEN
preprocessor = model_pipeline.named_steps["preprocessing"]
nieuwe_studenten_processed = preprocessor.transform(nieuwe_studenten)
nieuwe_studenten_processed

In [ ]:
# DEZE CODE GEBRUIK JE OM HET RESULTAAT TE TESTEN
feature_names = preprocessor.get_feature_names_out()
feature_names

### Opdracht 1.5.
Verwijder bij de ```model_pipeline``` de ```preprocessor``` en check wat het effect is op de code. Hoe komt dat?

In [ ]:
# DEZE CODE MOET JE AANPASSEN VOOR ELKE OPDRACHT

# Splits de kolommen in numerieke features en categoriale features omdat deze op een andere manier 'gepreprocessed' worden
numeric_features = ["leeftijd", "studie_uren"]
categorical_features = ["opleiding"]

# clean_opleiding is een custom functie voor de pipeline
def clean_opleiding(X):
    X = X.copy()
    X["opleiding"] = X["opleiding"].replace({
        "ict": "ICT"
    })
    return X

# Daarnaast maken we gebruik van een aantal standaard functies
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("clean_opleiding", FunctionTransformer(
        clean_opleiding,
        validate=False,
        feature_names_out="one-to-one"
    )),
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

# De pipeline wordt compleet gemaakt door de classifier
model_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", DecisionTreeClassifier(max_depth=3, random_state=42))
])

# De pipeline wordt uitgevoerd op de train data
model_pipeline.fit(X_train, y_train)

### Opdracht 1.6.
Verwijder bij de ```categorical_pipeline``` de ```FunctionTransformer``` en check wat het effect is op de getransformeerde data. Hoe komt dat?

In [ ]:
# DEZE CODE MOET JE AANPASSEN VOOR ELKE OPDRACHT

# Splits de kolommen in numerieke features en categoriale features omdat deze op een andere manier 'gepreprocessed' worden
numeric_features = ["leeftijd", "studie_uren"]
categorical_features = ["opleiding"]

# clean_opleiding is een custom functie voor de pipeline
def clean_opleiding(X):
    X = X.copy()
    X["opleiding"] = X["opleiding"].replace({
        "ict": "ICT"
    })
    return X

# Daarnaast maken we gebruik van een aantal standaard functies
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("clean_opleiding", FunctionTransformer(
        clean_opleiding,
        validate=False,
        feature_names_out="one-to-one"
    )),
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

# De pipeline wordt compleet gemaakt door de classifier
model_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", DecisionTreeClassifier(max_depth=3, random_state=42))
])

# De pipeline wordt uitgevoerd op de train data
model_pipeline.fit(X_train, y_train)

In [ ]:
# DEZE CODE GEBRUIK JE OM HET RESULTAAT TE TESTEN
preprocessor = model_pipeline.named_steps["preprocessing"]
nieuwe_studenten_processed = preprocessor.transform(nieuwe_studenten)
nieuwe_studenten_processed

In [ ]:
# DEZE CODE GEBRUIK JE OM HET RESULTAAT TE TESTEN
feature_names = preprocessor.get_feature_names_out()
feature_names

## Opdracht 2.

In deze opdrachten gaan we modellen trainen met een preprocessing pipeline.

We gebruiken de AmesHousing.csv in als pandas DataFrame. Deze heeft de volgende features: ```['LotFrontage', 'GrLivArea', 'GarageArea’,'Neighborhood', 'HouseStyle', 'ExterQual', 'MasVnrType’,’YearBuilt', 'YrSold', 'OverallQual']```
en de target ```SalePrice```.

### Opdracht 2.1
Maak een pipeline met de volgende onderdelen:

- Categoriale features
  - Neem enkel de categoriale feature ```Street```
  - Maak een ```ColumnTransformer``` met twee acties:
     1. Lege waardes opgevuld worden met de modus
     2. Waardes worden omgezet met de OneHotEncoder

- Numerieke features
  - Neem enkel de features ```GrLivArea``` en ```GarageArea```
  - Maak een ```ColumnTransformer``` met twee acties:
     1. Lege waardes opgevuld worden met de modus
     2. Normaliseren met de Z-score

- Model
  - Voeg een model toe aan de pipeline en train dat model met bovenstaande drie features
  - Evalueer het model op de testdata


### Opdracht 2.2
- Sla de hele pipeline op als joblib object
- Open het object in een nieuw notebook
- Run de pipeline met onderstaande data en check het of er daadwerkelijk voorspellingen worden gemaakt of dat er foutmeldingen komen
- Bekijk ook de processed data met behulp van onderstaande code en bekijk of de kolommen en inhoud er logisch uit zien:

    ```preprocessor = model_pipeline.named_steps["preprocessing"]```

    ```data_processed = preprocessor.transform(nieuwe_data)```)

### Opdracht 2.3

Breid de pipeline uit met extra features en denk goed na over welke pre processing nodig is voor elke feature.